# MCP task failure and recovery experiments
The local SQLite simulator injects transport and worker faults. These are captured simulator events, not production MCP or Kaggle failures.

## 1. Research scope
Recovery after interrupted tasks.
## 2. Isolated test environment
SQLite in memory; no production calls.
## 3. Task identity
Each case has a task ID and request key.
## 4. Durable state
Status, output and execution count are persisted.
## 5. Client-visible response
Fault injection can hide the durable state.
## 6. Original user task
Generate each artifact exactly once.
## 7. Network timeout
The task keeps running after a client timeout.
## 8. Restart with task ID
The client can poll the saved ID.
## 9. Restart without task ID
The client must search durable state or escalate.
## 10. Lost creation response
Creation may have succeeded before the response vanished.
## 11. Lost completion response
Output may already exist.
## 12. Worker crash after output
The failed task can still have an artifact.
## 13. Stale status
The cached status can disagree with the ledger.
## 14. Cancellation acknowledgement
Acknowledgement need not imply the operation stopped.
## 15. Duplicate request
The request key identifies existing work.
## 16. Existing task recovery
A recovered result can be reported.
## 17. Event provenance
Every scenario links to a captured simulator event.
## 18. Limited state policy
Reference actions use only visible information.
## 19. Full state policy
Reference actions also use verified ledger state.
## 20. Safe action sets
Multiple safe choices are explicit where appropriate.
## 21. Duplicate execution risk
Every case begins with one execution.
## 22. Trace export
Events and scenarios are saved as JSONL.
## 23. Deterministic capture
Run the next cell before LLM evaluation.

In [ ]:
from pathlib import Path
import sys, csv, json
PROJECT = Path.cwd()
if not (PROJECT / 'inference.py').exists():
    PROJECT = PROJECT / 'mcp_failures'
sys.path.insert(0, str(PROJECT))
from data import RESULTS, validate_protocol_traces
from inference import _read_csv
scenarios = [json.loads(line) for line in (RESULTS / 'llm_scenarios.jsonl').read_text(encoding='utf-8').splitlines()]
print(len(scenarios), 'preserved historical simulator scenarios')

## 24. OpenAI API configuration and connectivity test

In [ ]:
import inference
RUN_OPENAI = False  # Preserve the completed 1,800 calls; this phase makes no API requests.
OPENAI_BUDGET_USD = 5.0
OPENAI_MAX_OUTPUT_TOKENS = 300
OPENAI_RUN_MODE = 'pilot'
OPENAI_MODELS = ['gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.4']
inference.OPENAI_BUDGET_USD = OPENAI_BUDGET_USD
inference.OPENAI_MAX_OUTPUT_TOKENS = OPENAI_MAX_OUTPUT_TOKENS
inference.OPENAI_MODELS = OPENAI_MODELS
print('Historical Responses compatibility is documented in results/RESULTS.md.')
print('New OpenAI requests enabled:', RUN_OPENAI)

## 25. Prepare LLM recovery scenarios

In [ ]:
assert len(scenarios) == 60
print('Scenarios:', len(scenarios))
print('Sources:', sorted({s['provenance'] for s in scenarios}))
print('Failure modes:', sorted({s['failure_mode'] for s in scenarios}))

## 26. Evaluate GPT-5.4-mini

In [ ]:
mini_status = {'gpt-5.4-mini': 'completed historical run; no new calls'}
print(mini_status)

## 27. Evaluate GPT-5.4-nano, if available

In [ ]:
nano_status = {'gpt-5.4-nano': 'completed historical run; no new calls'}
print(nano_status)

## 28. Evaluate GPT-5.4

In [ ]:
large_status = {'gpt-5.4': 'completed historical run; no new calls'}
print(large_status)

## 29. Compare recovery decisions and unsafe retries

In [ ]:
raw = _read_csv(RESULTS / 'llm_inference_raw.csv')
assert len(raw) == 1800
comparison = _read_csv(RESULTS / 'llm_model_comparison.csv')
for row in comparison: print(row['model'], row['setting'], 'n=', row['n_valid'], 'accuracy=', row['accuracy'], 'unsafe retry=', row['unsafe_retry_rate'])

## 30. Analyze information availability versus model size

In [ ]:
for model in OPENAI_MODELS:
    pair = [r for r in comparison if r['model'] == model]
    print(model, {r['setting']: r['accuracy'] for r in pair})
print('Intervals in the comparison CSV use Wilson 95% bounds; repeated templates limit independence.')

## 31. Token usage, latency, and API cost

In [ ]:
for row in _read_csv(RESULTS / 'llm_cost_summary.csv'):
    print(row['model'], row['setting'], 'tokens=', row['input_tokens'], '+', row['output_tokens'], 'cost upper bound=', row['cost_usd_upper_bound'])

## 32. Final research findings and export

In [ ]:
print((RESULTS / 'RESULTS.md').read_text(encoding='utf-8'))
print('Handoff:', RESULTS / 'HANDOFF.md')


## 33. Actual MCP Tasks protocol validation
The official C# server smoke test is recorded in real_mcp_traces.jsonl. No API calls are made by this cell.

In [ ]:
check = validate_protocol_traces('protocol-042cb7301691')
assert check['exchanges'] == 32 and check['schema_checks'] == 55
print(check)
print((RESULTS / 'protocol_validation.md').read_text(encoding='utf-8'))

## 34. Real workload trial status and final handoff

In [ ]:
trials = _read_csv(RESULTS / 'real_mcp_trials.csv')
planned = _read_csv(RESULTS / 'real_mcp_summary.csv')
assert len(planned) == 48
assert len({row['trial_id'] for row in trials}) == len(trials)
print('Actual Instacart MCP workload trials:', sum(r['status'] == 'COMPLETED' for r in trials))
print('Unsupported combinations:', sum(r['status'] == 'NOT_RUN' for r in planned))
print('Handoff:', RESULTS / 'HANDOFF.md')

## 35. Real Instacart source validation
The competition URL is disabled; the four real CSVs came from the public Kaggle dataset psparks/instacart-market-basket-analysis through the official Kaggle API. Credentials are never displayed.

In [ ]:
manifest = json.loads((RESULTS / 'instacart_data_validation.json').read_text(encoding='utf-8'))
for name, item in manifest.items():
    if name == 'referential_integrity':
        continue
    assert (PROJECT / 'data' / name).is_file()
    assert item['malformed_rows'] == 0
    print(name, item['rows'], item['sha256'][:12])

## 36. Real Tasks workload outcomes
The saved rows came from the official C# SDK Tasks server with a custom Instacart tool. To run new trials, start server/InstacartMcp.csproj and use `python main.py --mode instacart-trials --repetitions 10`. This cell only audits saved results.

In [ ]:
from data import summarize_real_mcp_trials
audit = summarize_real_mcp_trials()
print(audit)
assert audit['trial_errors'] == 0
print((RESULTS / 'real_mcp_vs_simulator.md').read_text(encoding='utf-8'))